# Education 360° — Advanced Analytics
Explore call effectiveness, counselor performance, and combined operational indicators. Interpret relationships carefully and avoid name-based joins.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_PATH = Path.cwd().parent if Path.cwd().name.lower() == "python" else Path.cwd()
DATA_PATH = PROJECT_PATH / "Dataset"

# Accommodate the source filename spelling found in the project.
FILE_CANDIDATES = {
    "admissions": ["admissions.csv"],
    "branches dim": ["branches_dim.csv", "braches_dim.csv"],
    "calls": ["calls.csv"],
    "courses dim": ["courses_dim.csv"],
    "date table": ["DateTable.csv", "date_table.csv"],
    "employees": ["employees.csv"],
    "fees": ["fees.csv"],
    "leads": ["leads.csv"],
    "marketing": ["marketing.csv"],
    "placements": ["placements.csv"],
    "students dim": ["students_dim.csv"]
}

def load_table(table_name):
    candidates = FILE_CANDIDATES[table_name]
    for filename in candidates:
        path = DATA_PATH / filename
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(
        f"Could not find {table_name}. Tried: {candidates}. "
        f"Expected dataset folder: {DATA_PATH}"
    )

datasets = {}
for table in FILE_CANDIDATES:
    try:
        datasets[table] = load_table(table)
        print(f"{table:15} {datasets[table].shape}")
    except FileNotFoundError as exc:
        print(exc)

In [ ]:
calls = datasets["calls"].copy()
employees = datasets["employees"].copy()
leads = datasets["leads"].copy()
placements = datasets["placements"].copy()

## Call effectiveness

In [ ]:
display(calls["Call Category"].value_counts(dropna=False).rename_axis("Call Category").to_frame("Calls"))
effective = calls["Call Category"].astype(str).str.strip().str.lower().eq("effective call").sum()
print(f"Effective calls: {effective:,}")
print(f"Total calls: {len(calls):,}")
print(f"Effective call rate: {effective/len(calls)*100:.2f}%" if len(calls) else "No calls")

## Calls by counselor

In [ ]:
if "counselor id" in calls:
    counselor_calls = calls.groupby("counselor id").agg(
        Total_Calls=("call id","nunique"),
        Effective_Calls=("Call Category", lambda s: s.astype(str).str.strip().str.lower().eq("effective call").sum())
    ).reset_index()
    counselor_calls["Effective Rate %"] = counselor_calls["Effective_Calls"] / counselor_calls["Total_Calls"] * 100
    if {"emp id","emp name"}.issubset(employees.columns):
        counselor_calls = counselor_calls.merge(employees[["emp id","emp name"]],
                                                left_on="counselor id", right_on="emp id", how="left")
    display(counselor_calls.sort_values("Total_Calls", ascending=False))

## Counselor admissions and revenue

In [ ]:
admissions = datasets["admissions"].copy()
if {"counselor id","admission id","course fee"}.issubset(admissions.columns):
    counselor_admissions = admissions.groupby("counselor id").agg(
        Admissions=("admission id","nunique"),
        Revenue=("course fee","sum")
    ).reset_index()
    if {"emp id","emp name"}.issubset(employees.columns):
        counselor_admissions = counselor_admissions.merge(
            employees[["emp id","emp name"]], left_on="counselor id", right_on="emp id", how="left"
        )
    display(counselor_admissions.sort_values("Revenue", ascending=False))

## Cross-domain KPI summary

In [ ]:
fee = datasets["fees"]
pl = datasets["placements"]
ld = datasets["leads"]
summary = {
    "Admissions": len(admissions),
    "Leads": ld["lead id"].nunique(),
    "Converted Leads": ld["lead status"].astype(str).str.strip().str.lower().eq("converted").sum(),
    "Calls": len(calls),
    "Effective Calls": calls["Call Category"].astype(str).str.strip().str.lower().eq("effective call").sum(),
    "Placed Students": pl.loc[pl["placement status"].astype(str).str.strip().eq("Placed"),"student id"].nunique(),
    "Fees Billed": fee["total fee"].sum(),
    "Fees Collected": fee["paid fee"].sum()
}
display(pd.Series(summary, name="Value").to_frame())

## Limitations
- Lead and student names are not a reliable unique key; do not merge these tables by name.
- Marketing is an aggregate campaign table with a different grain.
- Validate date relationships and partial-year comparisons before drawing trend conclusions.